# 02 · Entity Mapping

**Purpose:** Map each article to the BVC tickers it mentions.

The RSS pipeline already ran LLM-based entity extraction and stored results in
`article_company_mentions`.  This notebook loads those LLM mentions and
**augments them with keyword matching** for articles that received no LLM tag.

**Input:** `articles.parquet`, `company_mentions.parquet`, `companies.parquet`  
**Output:** `articles_mapped.parquet`  
Schema: `article_url, published_at, date, ticker, title, text, language, feed_name, source`

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd

NB_DIR = Path().resolve()
sys.path.insert(0, str(NB_DIR))
from utils import load, save

In [2]:
articles         = load("articles")
company_mentions = load("company_mentions")
companies        = load("companies")

print(f"Articles:         {len(articles):,}")
print(f"Company mentions: {len(company_mentions):,}")
print(f"Tickers in DB:    {companies['ticker'].nunique()}")

FileNotFoundError: articles.parquet not found — run the earlier notebook first.

## 1 · LLM-derived mentions (primary source)

In [ ]:
ARTICLE_COLS = ["url", "published_at", "date", "title", "text", "language", "feed_name"]

llm_mapped = (
    company_mentions[["article_url", "ticker"]]
    .merge(articles[ARTICLE_COLS], left_on="article_url", right_on="url", how="inner")
    .drop(columns=["url"])
)
llm_mapped["source"] = "llm"

print(f"LLM-mapped rows:    {len(llm_mapped):,}")
print(f"Unique articles:    {llm_mapped['article_url'].nunique():,}")
print(f"Unique tickers:     {llm_mapped['ticker'].nunique()}")

## 2 · Keyword matching for un-mapped articles

In [ ]:
# Words that appear in many company names and would create false positives
_STOP = {"bank", "maroc", "groupe", "société", "compagnie", "holding",
         "africa", "maroc", "nationale", "general", "générale"}


def build_keyword_index(companies: pd.DataFrame) -> dict[str, str]:
    """Return {lowercase_keyword: ticker} mapping."""
    idx: dict[str, str] = {}
    for _, row in companies.iterrows():
        ticker = row["ticker"]
        name   = str(row["company_name"])

        # Always add the ticker itself
        idx[ticker.lower()] = ticker

        # Full company name
        idx[name.lower()] = ticker

        # Each meaningful word from the name (len > 4, not a stop word)
        for word in re.split(r"[\s\-/&()']+", name.lower()):
            if len(word) > 4 and word not in _STOP:
                idx.setdefault(word, ticker)  # first match wins for ambiguous words

    return idx


def match_tickers(text: str, kw_index: dict[str, str]) -> list[str]:
    text_lower = text.lower()
    return list({ticker for kw, ticker in kw_index.items() if kw in text_lower})


kw_index = build_keyword_index(companies)
print(f"Keyword index size: {len(kw_index):,} entries")

In [ ]:
already_mapped_urls = set(company_mentions["article_url"])
unmapped = articles[~articles["url"].isin(already_mapped_urls)].copy()
print(f"Un-mapped articles: {len(unmapped):,}")

unmapped["tickers"] = unmapped["text"].apply(lambda t: match_tickers(t, kw_index))
unmapped = unmapped.explode("tickers").dropna(subset=["tickers"])
unmapped = unmapped.rename(columns={"tickers": "ticker", "url": "article_url"})
unmapped["source"] = "keyword"
unmapped = unmapped[["article_url", "published_at", "date", "ticker",
                     "title", "text", "language", "feed_name", "source"]]

print(f"Keyword-matched rows: {len(unmapped):,}")
print(f"New unique articles:  {unmapped['article_url'].nunique():,}")

## 3 · Combine & deduplicate

In [ ]:
articles_mapped = (
    pd.concat([llm_mapped, unmapped], ignore_index=True)
    .drop_duplicates(subset=["article_url", "ticker"])
    .reset_index(drop=True)
)

print(f"Total mapped rows:  {len(articles_mapped):,}")
print(f"Unique articles:    {articles_mapped['article_url'].nunique():,}")
print(f"Unique tickers:     {articles_mapped['ticker'].nunique()}")
print()
print("Source breakdown:")
print(articles_mapped["source"].value_counts().to_string())

In [ ]:
# Most mentioned tickers
articles_mapped["ticker"].value_counts().head(15)

In [ ]:
save(articles_mapped, "articles_mapped")